In [1]:
from openai import OpenAI
import os 
api_key = os.getenv('api_key')
openai_client = OpenAI(api_key=api_key)

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
len(documents)

72

In [6]:
from minsearch import Index 

index = Index(text_fields=['content'],
              keyword_fields=['filename'])
index.fit(documents)

In [7]:
def search(question):
    boost_dict = {'filename': 1.0, 'content': 2.0}
    return index.search(question, boost_dict=boost_dict, num_results=5)

In [8]:
question = 'How does the agentic loop keep calling the model until it stops?'

In [9]:
search_results = search(question)

In [10]:
search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [11]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append('File name: ' + doc['filename'])
        lines.append('Content: ' +doc['content'])
        lines.append('')

    return '\n'.join(lines).strip()

In [12]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [13]:

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [14]:
model='gpt-5.4-mini'


In [15]:
def build_prompt(query, search_results):
    context = build_context(search_results)
    return PROMPT_TEMPLATE.format(
        question=query, context=context
    )

In [16]:
def llm(prompt):
    input_messages = [
        {'role': 'developer', 'content': INSTRUCTIONS},
        {'role': 'user', 'content': prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=input_messages
    )

    return response

In [17]:
search_results = search(question)
prompt = build_prompt(question, search_results)
response = llm(prompt)

In [18]:
print(response.usage.input_tokens)

7141


In [19]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [20]:
len(chunks)

295

In [21]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields= ["filename"]
)

chunk_index.fit(chunks)

In [22]:
def search(question):
    boost_dict = {'filename': 1.0, 'content': 2.0}
    return chunk_index.search(question, boost_dict=boost_dict, num_results=5)

In [23]:
search_results = search(question)
prompt = build_prompt(question, search_results)
response = llm(prompt)

In [24]:
print(response.usage.input_tokens)

2324


In [25]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [26]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [27]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'question': {'type': 'string',
     'description': 'question parameter'}},
   'required': ['question'],
   'additionalProperties': False}}]

In [28]:
AGENTIC_LOOP_INSTRUCTIONS = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

Make multiple searches.
"""


In [29]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)



In [30]:
from openai import OpenAI

client = OpenAI(api_key=os.getenv("api_key"))

llm_client = OpenAIClient(
    model="gpt-5.4-mini",
    client=client
)

In [31]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=AGENTIC_LOOP_INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=llm_client)

In [32]:
question = "How does the agentic loop work, and how is it different from plain RAG?"

result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received
